In [ ]:
import pyspark
from pyspark.sql import SparkSession

### Code for low memory

In [ ]:
import os
os.environ['SPARK_LOCAL_DIRS'] = '/tmp'

from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .master('local[1]') \
    .appName('zoomcamp-hw') \
    .config('spark.driver.memory', '512m') \
    .config('spark.driver.memoryOverhead', '256m') \
    .config('spark.sql.shuffle.partitions', '4') \
    .config('spark.ui.enabled', 'false') \
    .config('spark.sql.files.maxPartitionBytes', '16m') \
    .config('spark.memory.fraction', '0.6') \
    .config('spark.memory.storageFraction', '0.3') \
    .getOrCreate()

print('Spark started:', spark.version)

### Read the data 
Read with explicit mergeSchema=False — faster 
but may lead to missing columns if the schema is not consistent across files

In [ ]:
df = spark.read \
    .option('mergeSchema', 'false') \
    .parquet('yellow_tripdata_2025-11.parquet')

print(f'Count of partitions when reading: {df.rdd.getNumPartitions()}')
print(f'Number of columns: {len(df.columns)}')

In [ ]:
df.repartition(4) \
  .write \
  .mode('overwrite') \
  .parquet('yellow_2025_11_partitioned')

print('Task completed!')